# Two-call LangGraph workflow — walkthrough

Companion to `wiki/notebooks/langgraph-two-call-extension.md`. The graph code lives in
`two_call.py` (one source of truth); this notebook drives it step by step.
Everything runs offline first; the last section switches to a real model.

In [ ]:
# %pip install -q -r requirements.txt   # uncomment on first run
from two_call import (CritiqueState, ScriptedChat, build_classify_respond,
                      build_generate_critique, run_with_trace)
from langgraph.checkpoint.memory import InMemorySaver

## Step 1 — the State schema
Every key is a channel. `trace` has a reducer (`operator.add`), so every node appends to it.

In [ ]:
for key, annotation in CritiqueState.__annotations__.items():
    print(f"{key:15} {annotation}")

## Steps 2–4 — nodes, edges, compile
Build the pure two-call graph (`max_revisions=0`) on scripted replies and look at what compiled.

In [ ]:
PASS = '{"score": 8, "gaps": [], "failure_modes": [], "verdict": "pass"}'
chat = ScriptedChat(["KV cache blocks are allocated on demand...", PASS])
graph = build_generate_critique(chat, checkpointer=InMemorySaver())
print(graph.get_graph().draw_mermaid())

## Step 5 — run with an execution trace
Live per-node updates (`stream_mode="updates"`), then checkpoint history for the thread.

In [ ]:
final = run_with_trace(graph, {"question": "Why does vLLM page its KV cache?"}, thread_id="walkthrough-1")
final["critique"], final["trace"]

In [ ]:
config = {"configurable": {"thread_id": "walkthrough-1"}}
for snap in reversed(list(graph.get_state_history(config))):
    print(snap.metadata.get("step"), snap.next, sorted(snap.values))

## Step 6 — prove the handoff
The second call's user prompt must contain the first call's output.

In [ ]:
system, user = chat.calls[1]
assert "KV cache blocks are allocated on demand" in user
print(user)

## Step 7 — reflection loop
`max_revisions=1`: critique says revise → revise → critique again.

In [ ]:
REVISE = '{"score": 4, "gaps": ["no numbers"], "failure_modes": ["wrong flag"], "verdict": "revise"}'
loop = build_generate_critique(ScriptedChat(["v1", REVISE, "v2", PASS]), max_revisions=1,
                               checkpointer=InMemorySaver())
out = run_with_trace(loop, {"question": "q?"})
[e["node"] for e in out["trace"]], out["revisions"]

## Step 8 — variant B: classify → respond (conditional edge)

In [ ]:
triage = build_classify_respond(ScriptedChat([
    '{"topic": "billing", "sentiment": "negative", "urgency": "low"}', "Sorry about that — here's what happened..."]),
    checkpointer=InMemorySaver())
out = run_with_trace(triage, {"message": "Why did my GPU bill double?"})
out["classification"], out["responder"]

### Exercise — build variant C yourself
Explain → summarize: `concept → explanation → eli5`. The reference solution is in the wiki page;
try it first. Check: the summarize node's user prompt must equal the explanation.

In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class ExplainState(TypedDict, total=False):
    concept: str
    explanation: str
    eli5: str

# TODO: def explain(state) / def summarize(state), then wire START -> explain -> summarize -> END

## Step 9 — a real model
Pick a provider (`pip install langchain langchain-<provider>`, set its API key), or point at a
local vLLM server with `base_url`. Same graph, real calls — compare frontier vs local.

In [ ]:
# from two_call import make_chat
# chat = make_chat("<provider>:<model-name>")
# chat = make_chat("openai:<served-model-name>", base_url="http://localhost:8000/v1")   # local vLLM
# real = build_generate_critique(chat, max_revisions=1, checkpointer=InMemorySaver())
# run_with_trace(real, {"question": "How should I size --max-num-seqs for a 3B model on one GPU?"})